# Data Description
- subreddit: Post’s community name
- title: Post headline
- content: Post body text (your review data)
- score: Net upvote count
- create_utc: Post publish timestamp
- url: Unique post link for deduplication

Note: Duplicated posts captured by multiple keywords were deleted by URL deduplication.

In [ ]:
import praw
import pandas as pd
import os

cid = "uPQzbuH1tE4G8ALLmLxKuA"
csec = "V-sKVoO3yprn_hv2geKo2JU-jrrGfA"
uname = "zzxxzx-0032"

reddit = praw.Reddit(
    client_id=cid,
    client_secret=csec,
    user_agent=f"python:amazon_crawl:v1.0 (by /u/{uname})"
)
 # 26 key words
keywords = [
    "Rufus",
    "Rufus shopping assistant",
    "Amazon AI shopping assistant",
    "Amazon generative AI shopping",
    "Amazon shopping AI",
    "Alexa shopping assistant",
    "Alexa for Shopping",
    "Amazon product recommendation AI",
    "Amazon review summary AI",
    "Amazon AI search",
    "Amazon Rufus review",
    "Alexa AI shopping",
    "AI shopping assistant",
    "AI personal shopper",
    "AI product recommendation",
    "AI product comparison",
    "AI deal finder",
    "AI price comparison",
    "AI shopping agent",
    "AI shopping bot",
    "AI helped me buy",
    "ChatGPT shopping",
    "ChatGPT product recommendation",
    "AI shopping app",
    "AI shopping experience",
    "AI shopping review",
]

all_data = []
target_total = 2600   # goal number(can be changed)
per_key = 100        # single key word max number(can be changed)

for kw in keywords:
    print(f"key word: {kw}")
    # sort="new" 
    for post in reddit.subreddit("all").search(kw, sort="new", limit=per_key):
        all_data.append({
            "subreddit": post.subreddit.display_name,
            "title": post.title,
            "content": post.selftext,
            "score": post.score,
            "create_utc": post.created_utc,
            "url": f"https://reddit.com{post.permalink}"
        })
        # stop loop
        if len(all_data) >= target_total:
            break
    # stop crawl
    if len(all_data) >= target_total:
        break

# remove duplicates
df = pd.DataFrame(all_data).drop_duplicates(subset=["url"])

# save
desktop_path = os.path.expanduser("~/Desktop/Reddit_Amazon_Rufus.csv")
df.to_csv(desktop_path, encoding="utf-8-sig", index=False)

print(f"{len(df)} data")

key word: Rufus
key word: Rufus shopping assistant
key word: Amazon AI shopping assistant
key word: Amazon generative AI shopping
key word: Amazon shopping AI
key word: Alexa shopping assistant
key word: Alexa for Shopping
key word: Amazon product recommendation AI
key word: Amazon review summary AI
key word: Amazon AI search
key word: Amazon Rufus review
key word: Alexa AI shopping
key word: AI shopping assistant
key word: AI personal shopper
key word: AI product recommendation
key word: AI product comparison
key word: AI deal finder
key word: AI price comparison
key word: AI shopping agent
key word: AI shopping bot
key word: AI helped me buy
key word: ChatGPT shopping
key word: ChatGPT product recommendation
key word: AI shopping app
key word: AI shopping experience
key word: AI shopping review
1381 data


## Cleaning Data

In [63]:
import pandas as pd
import re
import os

file_path = os.path.expanduser("~/Desktop/Reddit_Amazon_Rufus.csv")
df = pd.read_csv(file_path)
origin_count = len(df)

# null
df = df.dropna(subset=["content"])
df["content"] = df["content"].astype(str).str.strip()
df = df[df["content"] != ""]

# remove URL multiples
df = df.drop_duplicates(subset=["url"], keep="first")

# cleaning
def clean_text(text):
    text = re.sub(r'http\S+|www.\S+', '', text)       # delete website
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)          # keep English & spaces
    text = text.lower()                               # small letters
    text = re.sub(r'\s+', ' ', text).strip()          # remove extra spaces
    return text

df["clean_content"] = df["content"].apply(clean_text)
df = df[df["clean_content"] != ""]
df = df[df["clean_content"].str.len()>3]

# save
desktop_path = os.path.expanduser("~/Desktop/cleaned_Reddit_Amazon_Rufus.csv")
df.to_csv(desktop_path, index=False, encoding="utf-8-sig")

print(f"original number: {origin_count}")
print(f"cleaning number: {len(df)}")

original number: 1381
cleaning number: 1284


## Comment Sentiment Classification
- positive
- negative
- mixed
- neutral
- empty/unknown

In [68]:
!pip install textblob
!python -m textblob.download_corpora

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 625.0/625.0 kB 7.9 MB/s eta 0:00:00
Finished.


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\l\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\brown.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\l\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\l\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\l\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to
[nltk_data]     C:\Users\l\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\conll2000.zip.
[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\l\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\movie_reviews.zip.


In [73]:
import pandas as pd
import os
from textblob import TextBlob

file_path = os.path.expanduser("~/Desktop/cleaned_Reddit_Amazon_Rufus.csv")
df = pd.read_csv(file_path)

df["sentiment_label"] = ""

def get_sentiment_label(text):
    # whether empty
    if pd.isna(text) or str(text).strip() == "":
        return "empty"
    try:
        blob = TextBlob(str(text))
        polarity = blob.sentiment.polarity
        subjectivity = blob.sentiment.subjectivity

        if polarity > 0.05:
            return "positive"
        elif polarity < -0.05:
            return "negative"
        else:
            return "neutral" if subjectivity < 0.3 else "mixed"
    except Exception as e:
        # unknown
        return "unknown"

df["sentiment_label"] = df["clean_content"].apply(get_sentiment_label)

# save
output_path = os.path.expanduser("~/Desktop/labeled_Reddit_Amazon_Rufus.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")

stat = df["sentiment_label"].value_counts()
print(stat)
print(f"\nempty：{stat.get('empty',0)}")
print(f"unknown：{stat.get('unknown',0)}")
print(f"valid comments：{df.shape[0] - stat.get('empty',0) - stat.get('unknown',0)}")

sentiment_label
positive    961
mixed       190
negative     69
neutral      64
Name: count, dtype: int64

empty：0
unknown：0
valid comments：1284
